In [ ]:
# read especData.dat in the output
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt

# Load the data from the file
with open('../../output/especData.dat', 'r') as f:
    lines = f.readlines()

# Process the data
i = 0
times = []
ke_data = []
pe_data = []
pe2_data = []
dke_data = []
has_pe2 = False
while i < len(lines):

    current_set = {}
    set_start = i
    
    while i < len(lines):
        line = lines[i].strip()
        if line:
            parts = line.split()
            label = parts[0].rstrip(':').upper()
            time_val = float(parts[1])
            data_vals = [float(x) for x in parts[2:]]
            
            current_set[label] = {'time': time_val, 'data': data_vals}
            i += 1
            
            # Check if we have a complete set (at minimum KE, PE, DKE)
            if 'KE' in current_set and 'PE' in current_set and 'DKE' in current_set:
                # Check if there's a PE2 in the next line
                if i < len(lines):
                    next_line = lines[i].strip()
                    if next_line and next_line.split()[0].rstrip(':').upper() == 'PE2':
                        continue  # Read PE2 as well
                break
        else:
            i += 1
    
    # Store the data from current set
    if 'KE' in current_set and 'PE' in current_set and 'DKE' in current_set:
        times.append(current_set['KE']['time'])
        ke_data.append(current_set['KE']['data'])
        pe_data.append(current_set['PE']['data'])
        dke_data.append(current_set['DKE']['data'])
        
        # Check for PE2
        if 'PE2' in current_set:
            pe2_data.append(current_set['PE2']['data'])
            has_pe2 = True
        elif has_pe2:  # If we had PE2 before but not now, pad with zeros
            pe2_data.append([0.0] * len(current_set['PE']['data']))

ke_data = np.array(ke_data)
pe_data = np.array(pe_data)
dke_data = np.array(dke_data)
times = np.array(times)

if has_pe2:
    pe2_data = np.array(pe2_data)

# Calculate total energy
total_ke = np.sum(ke_data, axis=1)*0.5
total_pe = np.sum(pe_data, axis=1)
total_pe = total_pe - total_pe[0]  # Normalize potential energy

if has_pe2:
    total_pe2 = np.sum(pe2_data, axis=1)
    total_pe2 = total_pe2 - total_pe2[0]  # Normalize potential energy

total_energy = total_ke + total_pe

# Calculate time step sizes and integrate dKE properly
time_steps = np.diff(times)
time_steps = np.append(time_steps[0], time_steps)  # Use first time step for initial value
total_dke_per_step = np.sum(dke_data, axis=1) * time_steps
total_dke = np.cumsum(total_dke_per_step)

# Print information about detected data
print(f"Detected data types: KE, PE, DKE", end="")
if has_pe2:
    print(", PE2")
else:
    print()
print(f"Number of time steps: {len(times)}")

fig, axes = plt.subplots(1, 3, figsize=(18, 12))
fig.suptitle('Energy Analysis Dashboard', fontsize=16, fontweight='bold')

# 1. Energy Evolution Over Time
ax1 = axes[0]
ax1.plot(times, total_ke, label='Kinetic Energy (KE)', linestyle='-', linewidth=2, color='blue')
ax1.plot(times, total_pe, label='Potential Energy (PE)', linestyle='-', linewidth=2, color='red')
ax1.plot(times, total_energy, label='Total Energy (KE + PE)', linestyle='--', linewidth=2, color='black')
ax1.set_xlabel('Time')
ax1.set_ylabel('Energy')
ax1.set_title('Energy Evolution Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. dKE Analysis
ax2 = axes[1]
ax2.plot(times, total_dke, label='Cumulative dKE', linestyle='-', linewidth=2, color='green')
ax2.plot(times, total_ke-total_ke[0], label='ΔKE = KE - KE(0)', linestyle='--', linewidth=2, color='blue')
difference = (total_ke - total_ke[0]) - total_dke
ax2.plot(times, difference, label='Difference: ΔKE - Cumulative dKE', linestyle=':', linewidth=2, color='red')
ax2.set_xlabel('Time')
ax2.set_ylabel('Energy')
ax2.set_title('dKE vs KE Change Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Energy Difference (Residual)
ax3 = axes[2]
ax3.plot(times, np.abs(difference), linewidth=2, color='red')
ax3.set_xlabel('Time')
ax3.set_ylabel('Energy Difference')
ax3.set_title('Residual: ΔKE - Cumulative dKE')
ax3.grid(True, alpha=0.3)
ax3.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

# Time stamp table
detailed_data = []
for i, t in enumerate(times):
    row = {
        'Time': f'{t:.4f}',
        'KE': f'{total_ke[i]:.6e}',
        'PE': f'{total_pe[i]:.6e}',
        'Cumul. dKE': f'{total_dke[i]:.6e}',
        'Total Energy': f'{total_energy[i]:.6e}',
        'ΔKE': f'{total_ke[i] - total_ke[0]:.6e}',
        'Residual': f'{(total_ke[i] - total_ke[0]) - total_dke[i]:.6e}'
    }
    if has_pe2:
        row['PE2'] = f'{total_pe2[i]:.6e}'
    detailed_data.append(row)

df = pd.DataFrame(detailed_data)

styled_df = df.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10px'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#40466e'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

print("\n" + "="*60)
print("ENERGY ANALYSIS SUMMARY")
print("="*60)
print(f"Initial KE: {total_ke[0]:.6e}")
print(f"Final KE: {total_ke[-1]:.6e}")
print(f"KE Change: {total_ke[-1] - total_ke[0]:.6e}")
print(f"Cumulative dKE: {total_dke[-1]:.6e}")
print(f"Residual (ΔKE - Cum.dKE): {difference[-1]:.6e}")
print(f"Initial PE: {total_pe[0]:.6e}")
print(f"Final PE: {total_pe[-1]:.6e}")
print(f"PE Change: {total_pe[-1] - total_pe[0]:.6e}")
print(f"Total Energy Change: {total_energy[-1] - total_energy[0]:.6e}")
if has_pe2:
    print(f"PE2 Change: {total_pe2[-1] - total_pe2[0]:.6e}")
print("="*60)
print("DETAILED ENERGY EVOLUTION TABLE")
print("="*60)
print(f"Total time steps: {len(times)}")
display(styled_df)